# Cross-Country Development Analytics: GDP and Child Mortality

An end-to-end pandas pipeline over four global datasets (180+ countries,
1800-2020): clean the four panels, measure GDP recovery to 90% of 2010 levels,
model child mortality from GDP for each country, and examine where a simple
linear model breaks down.

**Stack:** Python - pandas - NumPy - scikit-learn

## 1. Loading the data

Four longitudinal panels, each in long format (`country`, `time`, value):

- `child_mortality_0_5_year_olds_dying_per_1000_born.csv` - under-5 child mortality (deaths per 1,000 live births)
- `co2_pcap_cons.csv` - CO2 emissions per capita (tonnes)
- `gdp_pcap.csv` - GDP per capita (2021 USD)
- `mincpcap_cppp.csv` - inflation-adjusted average daily income (USD/person/day)

In [1]:
# Load the four panels into separate DataFrames.
import pandas as pd

df1 = pd.read_csv('child_mortality_0_5_year_olds_dying_per_1000_born.csv')
df2 = pd.read_csv('co2_pcap_cons.csv')
df3 = pd.read_csv('gdp_pcap.csv')
df4 = pd.read_csv('mincpcap_cppp.csv')

# Inspect dtypes, row counts and missing values in each panel.
print(df1.info())
print(df2.info())
print(df3.info())
print(df4.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55536 entries, 0 to 55535
Data columns (total 3 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   country                                            55536 non-null  object 
 1   time                                               55536 non-null  int64  
 2   child_mortality_0_5_year_olds_dying_per_1000_born  55531 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 1.3+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41255 entries, 0 to 41254
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   country        41255 non-null  object 
 1   time           41255 non-null  int64  
 2   co2_pcap_cons  41252 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 967.0+ KB
None
<class 'pandas.core.frame.Da

## 2. Cleaning the data

Some entries are invalid - empty (`NaN`), zero, or negative - and some countries
are missing entirely from a given panel. The cleaning rule:

- Replace each missing/invalid value with the **next valid year** within the same country.
- Drop any country absent from one or more panels, so all four panels stay aligned.

The cleaned panels are written to new `*_clean.csv` files; the originals are left untouched.

In [2]:
import numpy as np

# Count missing values in each panel before cleaning.
print(df1.isna().sum())
print('-----')
print(df2.isna().sum())
print('-----')
print(df3.isna().sum())
print('-----')
print(df4.isna().sum())
print('-----')

# Replace invalid values (<= 0) with NaN, then back-fill each gap with the
# next valid year within the same country.
cols1 = df1.select_dtypes(include=[np.number]).columns
df1[cols1] = df1[cols1].mask(df1[cols1] <= 0)
df1 = df1.sort_values(by=['country','time'])
df1[cols1] = df1.groupby('country')[cols1].bfill()

cols2 = df2.select_dtypes(include=[np.number]).columns
df2[cols2] = df2[cols2].mask(df2[cols2] <= 0)
df2 = df2.sort_values(by=['country','time'])
df2[cols2] = df2.groupby('country')[cols2].bfill()

cols3 = df3.select_dtypes(include=[np.number]).columns
df3[cols3] = df3[cols3].mask(df3[cols3] <= 0)
df3 = df3.sort_values(by=['country','time'])
df3[cols3] = df3.groupby('country')[cols3].bfill()

cols4 = df4.select_dtypes(include=[np.number]).columns
df4[cols4] = df4[cols4].mask(df4[cols4] <= 0)
df4 = df4.sort_values(by=['country','time'])
df4[cols4] = df4.groupby('country')[cols4].bfill()

# Keep only countries present in all four panels, so the panels stay aligned.
common_countries = set(df1['country']) & set(df2['country']) & set(df3['country']) & set(df4['country'])
df1=df1[df1['country'].isin(common_countries)]
df2=df2[df2['country'].isin(common_countries)]
df3=df3[df3['country'].isin(common_countries)]
df4=df4[df4['country'].isin(common_countries)]

# Save the cleaned panels to new *_clean.csv files (originals are left untouched).
df1.to_csv('child_mortality_0_5_year_olds_dying_per_1000_born_clean.csv',index=False)
df2.to_csv('co2_pcap_cons_clean.csv',index=False)
df3.to_csv('gdp_pcap_clean.csv',index=False)
df4.to_csv('mincpcap_cppp_clean.csv',index=False)

country                                              0
time                                                 0
child_mortality_0_5_year_olds_dying_per_1000_born    5
dtype: int64
-----
country          0
time             0
co2_pcap_cons    3
dtype: int64
-----
country     0
time        0
gdp_pcap    6
dtype: int64
-----
country          0
time             0
mincpcap_cppp    3
dtype: int64
-----


## 3. When did each economy recover to 90% of its 2010 GDP?

For each country, find the first year its GDP per capita reached 90% of its 2010
value, then report the mean of those years across all countries.

In [3]:
# Load the cleaned panels produced in the previous step.
ndf1 = pd.read_csv('child_mortality_0_5_year_olds_dying_per_1000_born_clean.csv')
ndf2 = pd.read_csv('co2_pcap_cons_clean.csv')
ndf3 = pd.read_csv('gdp_pcap_clean.csv')
ndf4 = pd.read_csv('mincpcap_cppp_clean.csv')

# For each country, compute 90% of its 2010 GDP as the recovery threshold.
gdp_2010=ndf3[ndf3['time']==2010][['country','gdp_pcap']]
gdp_2010['threshold'] = gdp_2010['gdp_pcap']*0.9
target_data = gdp_2010[['country','threshold']]

# Attach each country's threshold back onto its full GDP history.
merged_df = pd.merge(ndf3,target_data,on='country')
# Keep the years where GDP is at or above the 90% threshold.
qualified = merged_df[merged_df['gdp_pcap']>= merged_df['threshold']]

first_year=qualified.groupby('country')['time'].min()
mean_year=first_year.mean()


# Print the mean recovery year across countries (rounded to the nearest integer).
print(round(mean_year))

1993


## 4. Per-country model: predicting 2020 child mortality from GDP

For each country separately, fit a simple linear regression that predicts child
mortality from GDP, training on every fifth year from 1800-2010, then predict 2020.
Predictions are saved to `predicted_child_mortality.csv`, and we report the five
countries whose models predict 2020 most accurately.

In [4]:
from sklearn.linear_model import LinearRegression

# Build the modelling table (child mortality + GDP), then take the training set:
# every fifth year from 1800 to 2010.
full_data=pd.merge(ndf1,ndf3,on=['country','time'])
train_data=full_data[(full_data['time']>=1800) & (full_data['time']<=2010) & (full_data['time']%5==0)]

all_2020_data=full_data[full_data['time']==2020]
countries=full_data['country'].unique()

# Fit one linear regression per country, then predict that country's 2020 mortality.
result=[]
for country in countries:
    country_data=train_data[train_data['country']==country]
    country_2020=all_2020_data[all_2020_data['country']==country]

    # Skip countries with no training rows or no 2020 record.
    if len(country_data)==0 or len(country_2020)==0:
        continue

    gdp=country_data[['gdp_pcap']]
    mortality=country_data['child_mortality_0_5_year_olds_dying_per_1000_born']
    pre2020=country_2020[['gdp_pcap']]

    lin=LinearRegression()
    lin.fit(gdp,mortality)

    prediction=lin.predict(pre2020)[0]

    true_val=country_2020['child_mortality_0_5_year_olds_dying_per_1000_born'].values[0]

    result.append({'country':country,'prediction_child_mortality':prediction,"true_value":true_val})

# Save the per-country predictions to a CSV file.
result_df=pd.DataFrame(result)
result_df[['country', 'prediction_child_mortality']].to_csv('predicted_child_mortality.csv')

# Rank countries by absolute error and report the five closest predictions.
result_df['error'] = abs(result_df['prediction_child_mortality'] - result_df['true_value'])
top5 = result_df.sort_values('error').head(5)
print("top5 closest prediction：", top5['country'].tolist())

#print(result_df)

top5 closest prediction： ['tcd', 'omn', 'sdn', 'mwi', 'jor']


## 5. Why some 2020 predictions are negative

A handful of countries get a negative predicted mortality for 2020, which is
impossible. The note below explains why this happens and how it could be fixed.

The main reason is that the relationship between GDP and child mortality is not
linear. Over the 200-year history, many countries move from low GDP (where
mortality drops fast) to high GDP (where mortality flattens out near 0, but never
goes below 0). Linear regression, however, fits a straight line: it "remembers"
the steep historical drop and extends it to the present, producing negative
predictions for 2020.

A simple fix is to take the log of GDP before training, which makes the
relationship more linear.